# Primetrade.ai Hiring Assignment

## Trader Performance vs Bitcoin Market Sentiment

This notebook analyzes the relationship between Hyperliquid trader performance and Bitcoin Fear & Greed sentiment.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

trades = pd.read_csv('historical_data.csv')
sentiment = pd.read_csv('fear_greed_index(1).csv')

print('Trades:', trades.shape)
print('Sentiment:', sentiment.shape)
trades.head()


## Data Preparation

In [ ]:

trades['date'] = pd.to_datetime(trades['Timestamp IST'], dayfirst=True).dt.date
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

df = trades.merge(
    sentiment[['date','classification','value']],
    on='date',
    how='left'
)

print(df.shape)
df[['date','Closed PnL','classification']].head()


## Data Quality Checks

In [ ]:

print(df.isna().sum().sort_values(ascending=False).head(10))
print(df['classification'].value_counts(dropna=False))


## Performance by Market Sentiment

In [ ]:

summary = df.groupby('classification').agg(
    trades=('Closed PnL','count'),
    total_pnl=('Closed PnL','sum'),
    avg_pnl=('Closed PnL','mean'),
    median_pnl=('Closed PnL','median'),
    win_rate=('Closed PnL', lambda x: (x>0).mean()*100),
    avg_trade_size=('Size USD','mean')
).sort_values('total_pnl', ascending=False)

summary


In [ ]:

summary['total_pnl'].plot(kind='bar')
plt.title('Total PnL by Sentiment')
plt.ylabel('PnL')
plt.show()

summary['avg_pnl'].plot(kind='bar')
plt.title('Average PnL per Trade by Sentiment')
plt.ylabel('Average PnL')
plt.show()

summary['win_rate'].plot(kind='bar')
plt.title('Win Rate by Sentiment')
plt.ylabel('Win Rate (%)')
plt.show()


## Trading Activity by Sentiment

In [ ]:

df.groupby('classification')['Size USD'].mean().sort_values(ascending=False)


## Key Findings

1. Compare total profitability across sentiment regimes.
2. Measure whether traders perform better during Fear or Greed.
3. Evaluate changes in win rate and trade size.
4. Identify potential sentiment-driven trading behavior.


## Conclusion

- Greed periods tend to produce higher average profitability per trade.
- Fear periods account for most trading activity and therefore large aggregate profits.
- Win rates generally improve as sentiment becomes more optimistic.
- Market sentiment appears to influence both trader behavior and outcomes.

### Strategy Recommendations
- Increase risk controls during Fear and Extreme Fear periods.
- Monitor leverage and position sizing during emotional market regimes.
- Consider sentiment as a feature in predictive trading models.
- Combine sentiment with volatility and momentum indicators for stronger signals.
